# Message brokers — AMQP 1.0 in-process, NATS and Kafka by description

NATS and Kafka need a real broker, so this notebook runs the AMQP 1.0 adapter against `AmqpMiniBroker`, a small in-process broker for tests and demos (not for production). The NATS and Kafka adapters follow the same endpoint model; their usage is shown in text below.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.22.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.Amqp, 0.22.0-preview.1"

## A broker and a gateway
The mini broker keeps an in-memory queue per address and checks SASL PLAIN credentials when you give it some. Credentials are secrets: the ones here are demo values.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Adapters.Amqp;
using System.Text;

var broker = AmqpMiniBroker.Create(userName: "plant", password: "demo-only");
await broker.StartAsync();
var gateway = AmqpEndpoint.Create(o => o.UseBroker(broker.Address).WithCredentials("plant", "demo-only"));
await gateway.ConnectAsync();
Console.WriteLine($"connected: {gateway.IsConnected}");

for (var i = 0; i < 3; i++)
    await gateway.PublishAsync("plant.temp", Encoding.UTF8.GetBytes($"2{i}.5"), new PublishOptions { QualityOfService = QualityOfService.AtLeastOnce, ContentType = "text/plain" });
Console.WriteLine($"waiting on plant.temp: {broker.QueueLength("plant.temp")}");

## Subscribe
A subscription is a receiver link. Each message is accepted when the loop asks for the next one.

In [ ]:
var cts = new CancellationTokenSource(TimeSpan.FromSeconds(10));
var got = new List<string>();
await foreach (var m in gateway.SubscribeAsync("plant.temp", cts.Token))
{
    got.Add(Encoding.UTF8.GetString(m.Payload.Span));
    if (got.Count == 3) break;
}
Console.WriteLine(string.Join(", ", got));

## Redelivery and rejection
A message the consumer never finished with goes back to the queue. A broker that rejects a message makes a confirmed publish throw; a pre-settled publish does not wait for the outcome.

In [ ]:
await gateway.PublishAsync("plant.alarm", "overheat"u8.ToArray(), new PublishOptions { QualityOfService = QualityOfService.AtLeastOnce });
await foreach (var m in gateway.SubscribeAsync("plant.alarm", cts.Token))
{
    Console.WriteLine($"took {Encoding.UTF8.GetString(m.Payload.Span)} and stopped without finishing");
    break;
}
await foreach (var m in gateway.SubscribeAsync("plant.alarm", cts.Token))
{
    Console.WriteLine($"delivered again: {Encoding.UTF8.GetString(m.Payload.Span)}");
    break;
}

broker.RejectedAddresses.Add("plant.closed");
try { await gateway.PublishAsync("plant.closed", "x"u8.ToArray(), new PublishOptions { QualityOfService = QualityOfService.AtLeastOnce }); }
catch (DeviceException e) { Console.WriteLine("confirmed publish failed: " + e.Message); }
await gateway.PublishAsync("plant.closed", "x"u8.ToArray());   // pre-settled: no outcome awaited
Console.WriteLine($"messages accepted by consumers so far: {broker.MessagesDelivered}");

## Wrong credentials
The broker checks them, and the failure is a `TransportException`.

In [ ]:
var intruder = AmqpEndpoint.Create(o => o.UseBroker(broker.Address).WithCredentials("plant", "wrong"));
try { await intruder.ConnectAsync(); } catch (TransportException e) { Console.WriteLine("refused: " + e.GetType().Name); }
await intruder.DisposeAsync();
await gateway.DisposeAsync();
await broker.DisposeAsync();

## NATS and Kafka
These need a real broker, so they are shown as text. Both are endpoints like the one above (`ConnectAsync`, `PublishAsync`, `SubscribeAsync`, traffic tap, hosting helpers `AddNats` and `AddKafka`).

```csharp
// NATS: wildcards, queue groups and request/reply
using IoTCom.Net.Adapters.Nats;

await using var nats = NatsEndpoint.Create(o => o.UseServer("nats://localhost:4222"));
await nats.ConnectAsync();
await nats.PublishAsync("plant.line1.temp", "21.5"u8.ToArray());
await foreach (var m in nats.ReceiveAsync("plant.>", queueGroup: "loggers")) { /* each message goes to one group member */ }
var reply = await nats.RequestAsync("plant.line1.info", ReadOnlyMemory<byte>.Empty);
```

```csharp
// Kafka: keys, headers, consumer groups, regular-expression topics (filters starting with ^)
using IoTCom.Net.Adapters.Kafka;

await using var kafka = KafkaEndpoint.Create(o => o.UseBootstrap("localhost:9092").WithGroup("gateway"));
await kafka.ConnectAsync();
await kafka.PublishAsync("plant-temp", key: "line1"u8.ToArray(), value: "21.5"u8.ToArray());
await foreach (var r in kafka.ReceiveAsync("^plant-.*")) Console.WriteLine($"{r.Topic}[{r.Partition}]@{r.Offset}");
```

Their integration tests run when `IOTCOM_NATS_URL` and `IOTCOM_KAFKA_BOOTSTRAP` are set.

## Going further
See `docs/en/protocols/messaging-brokers.md` and `docs/en/protocols/zenoh.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*